# Proyecto 2 Entrega de Ingeniería de datos

Trabajamos con los anuncios de Airbnb de San Diego. Preparamos una base para comparar alojamientos y otra con precios publicados por noche para el análisis de regresión. Cada fila corresponde a un anuncio.

## 0. Librerías y carga de datos

Usamos el archivo detallado de Inside Airbnb que se descargó.

In [2]:
from pathlib import Path
import hashlib
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

carpeta_actual = Path.cwd()
raiz = carpeta_actual if (carpeta_actual / "listings.csv").is_file() else carpeta_actual.parent
entrada = raiz / "listings.csv"
if not entrada.is_file():
    raise FileNotFoundError("Abre el notebook desde el proyecto o su carpeta ingenieria_datos.")

salida = raiz / "ingenieria_datos" / "datos_procesados"
ids_texto = {c: "string" for c in ["id", "host_id", "host_profile_id", "scrape_id"]}
marcadores = ["", "n", "na", "n/a", "nan", "none", "null"]
original = pd.read_csv(entrada, dtype=ids_texto, low_memory=False,
                       na_values=["N", "n", "N/A", "NULL", "null", "None"])
print(f"{entrada.name}: {original.shape[0]:,} anuncios y {original.shape[1]} columnas.")
print("Capturas:", original["last_scraped"].min(), "a", original["last_scraped"].max())
display(original[["neighbourhood_cleansed", "room_type", "accommodates", "price"]].head())

listings.csv: 13,253 anuncios y 90 columnas.
Capturas: 2026-09-21 a 2026-09-23


,neighbourhood_cleansed,room_type,accommodates,price
0,North Hills,Entire home/apt,7,$316.00
1,Pacific Beach,Private room,2,$146.50
2,Pacific Beach,Private room,2,$121.00
3,North Hills,Private room,2,NaN
4,Pacific Beach,Entire home/apt,2,$282.00


## 1. Exploración inicial

Revisamos los tipos de datos y contamos los valores que faltan en cada columna. También miramos cuántos anuncios tienen precio por tipo de habitación.

In [3]:
perfil = pd.DataFrame({
    "tipo_leído": original.dtypes.astype(str),
    "nulos": original.isna().sum(),
    "nulos_%": original.isna().mean().mul(100).round(2),
    "valores_distintos": original.nunique()
}).sort_values("nulos", ascending=False, kind="stable")
print(f"Columnas completamente vacías: {original.isna().all().sum()}")
with pd.option_context("display.max_rows", len(perfil)):
    display(perfil)

display(original[["latitude", "longitude", "accommodates", "bedrooms", "beds",
                  "minimum_nights", "availability_365", "number_of_reviews",
                  "review_scores_rating"]].describe().T.round(2))

Columnas completamente vacías: 14


,tipo_leído,nulos,nulos_%,valores_distintos
neighborhood_overview,float64,13253,100.00,0
host_since,float64,13253,100.00,0
host_response_time,float64,13253,100.00,0
host_response_rate,float64,13253,100.00,0
host_acceptance_rate,float64,13253,100.00,0
host_thumbnail_url,float64,13253,100.00,0
host_neighbourhood,float64,13253,100.00,0
host_total_listings_count,float64,13253,100.00,0
host_verifications,float64,13253,100.00,0
neighbourhood,float64,13253,100.00,0


,count,mean,std,min,25%,50%,75%,max
latitude,13253.0,32.77,0.07,32.54,32.73,32.76,32.80,33.09
longitude,13253.0,-117.18,0.07,-117.28,-117.25,-117.17,-117.14,-116.94
accommodates,13253.0,5.15,3.37,1.00,2.00,4.00,6.00,16.00
bedrooms,13229.0,1.96,1.42,0.00,1.00,2.00,3.00,32.00
beds,12880.0,2.89,2.37,1.00,1.00,2.00,4.00,37.00
minimum_nights,13171.0,11.80,21.39,1.00,2.00,2.00,31.00,730.00
availability_365,13253.0,217.94,121.64,0.00,106.00,252.00,329.00,365.00
number_of_reviews,13253.0,73.78,120.45,0.00,3.00,25.00,93.00,1918.00
review_scores_rating,11099.0,4.81,0.33,1.00,4.77,4.90,4.98,5.00


In [4]:
cobertura = original.groupby("room_type")["price"].agg(anuncios="size", con_precio="count")
cobertura["sin_precio"] = cobertura["anuncios"] - cobertura["con_precio"]
cobertura["cobertura_%"] = (100 * cobertura["con_precio"] / cobertura["anuncios"]).round(1)
display(cobertura)
display(original.groupby(original["price"].notna().map({True: "Con precio", False: "Sin precio"}))
        [["availability_365", "number_of_reviews"]].median()
        .rename_axis("Precio publicado").rename(columns={"availability_365": "Mediana días disponibles",
                                                         "number_of_reviews": "Mediana reseñas"}))

,anuncios,con_precio,sin_precio,cobertura_%
room_type,,,,
Entire home/apt,11052,9764,1288,88.3
Hotel room,277,267,10,96.4
Private room,1872,1541,331,82.3
Shared room,52,42,10,80.8


,Mediana días disponibles,Mediana reseñas
Precio publicado,,
Con precio,264.0,30.0
Sin precio,0.0,6.0


## 2. Selección de variables

Para estudiar el precio usamos el barrio, el tipo de alojamiento, la capacidad y los servicios. Guardamos las reseñas y los días disponibles para comparar los anuncios.

Quitamos nombres, enlaces y columnas vacías. También dejamos por fuera los ingresos estimados porque pueden revelar el precio.

In [5]:
columnas = [
    "id", "host_id", "last_scraped", "neighbourhood_cleansed", "room_type", "property_type",
    "latitude", "longitude", "accommodates", "bedrooms", "beds", "bathrooms_text", "amenities",
    "host_is_superhost", "minimum_nights", "availability_365", "number_of_reviews",
    "last_review", "review_scores_rating", "price"
]
d = original[columnas].copy()
# Quitamos las filas que se repiten por completo.
d = d.loc[~original.duplicated()].copy()
if d["id"].isna().any() or not d["id"].is_unique:
    raise ValueError("Revisar registros sin ID o con el mismo ID antes de consolidar el corte.")
if d["host_id"].isna().any():
    raise ValueError("Falta el anfitrión necesario para la partición por grupos.")

## 3. Limpieza de los datos

Pasamos los textos a minúsculas y quitamos los espacios de más. Contamos cada medio baño como 0,5 y marcamos los servicios que menciona el anuncio.

In [6]:
def texto_uniforme(serie):
    texto = serie.astype("string").str.normalize("NFKC").str.strip().str.lower()
    texto = texto.str.replace(r"\s+", " ", regex=True)
    return texto.mask(texto.isin(marcadores))

base = d[["id", "host_id"]].copy()
categoricas = ["barrio", "tipo_habitacion", "tipo_propiedad", "privacidad_bano"]
for fuente, destino in {"neighbourhood_cleansed": "barrio", "room_type": "tipo_habitacion",
                        "property_type": "tipo_propiedad"}.items():
    base[destino] = texto_uniforme(d[fuente]).fillna("sin_especificar")

bano = texto_uniforme(d["bathrooms_text"])
cantidad = pd.to_numeric(bano.str.extract(r"(\d+(?:\.\d+)?)", expand=False), errors="coerce")
base["banos"] = cantidad.mask(bano.str.contains("half", na=False), 0.5).astype("float64")
base["privacidad_bano"] = np.select(
    [bano.str.contains("shared", na=False).to_numpy(dtype=bool),
     bano.str.contains("private", na=False).to_numpy(dtype=bool)],
    ["compartido", "privado"], default="sin_especificar")
display(pd.DataFrame({
    "Baño en el anuncio": d["bathrooms_text"],
    "Baños": base["banos"],
    "Privacidad": base["privacidad_bano"]
}).drop_duplicates().head())

,Baño en el anuncio,Baños,Privacidad
0,2 baths,2.0,sin_especificar
1,1 private bath,1.0,privado
4,1 bath,1.0,sin_especificar
13,1 shared bath,1.0,compartido
36,3 baths,3.0,sin_especificar


In [7]:
def servicios_de_anuncio(valor):
    # El anuncio no indica servicios.
    if pd.isna(valor):
        servicios = []
    else:
        servicios = json.loads(valor)
        if not isinstance(servicios, list) or not all(isinstance(s, str) for s in servicios):
            raise ValueError("Se esperaba una lista de servicios en amenities.")
    servicios = {s.casefold().strip() for s in servicios if s.strip()}
    return {
        "servicios_declarados": len(servicios),
        "declara_wifi": int(any(re.search(r"\bwi[ -]?fi\b", s) for s in servicios)),
        "declara_cocina": int(bool(servicios & {"kitchen", "kitchenette"})),
        "declara_parqueadero_gratis_predio": int(any(
            s.startswith("free ") and "parking" in s and "on premises" in s for s in servicios))
    }

servicios = pd.DataFrame(d["amenities"].map(servicios_de_anuncio).tolist(), index=d.index)
base = base.join(servicios)
binario = texto_uniforme(d["host_is_superhost"]).map({"t": 1, "f": 0})
if binario.isna().any():
    raise ValueError("Revisar los valores originales de host_is_superhost.")
base["superanfitrion"] = binario.astype("int8")
servicios_por_anuncio = base[["declara_wifi", "declara_cocina", "declara_parqueadero_gratis_predio"]]
display(servicios_por_anuncio.sum().set_axis(["Wifi", "Cocina", "Parqueadero gratis en el alojamiento"])
        .to_frame("Anuncios que lo ofrecen"))

,Anuncios que lo ofrecen
Wifi,13130
Cocina,11919
Parqueadero gratis en el alojamiento,8759


In [8]:
# Convertimos $1,200.00 en 1200.00.
texto_precio = d["price"].astype("string").str.strip()
patron_usd = r"\$?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{1,2})?"
importe = texto_precio.where(texto_precio.str.fullmatch(patron_usd, na=False))
precio = pd.to_numeric(importe.str.replace(r"[$,]", "", regex=True), errors="coerce").astype("float64")
precio = precio.where(precio.gt(0)).rename("precio_noche_usd")
monedas = (original.loc[d.index, "price_quote_raw"].dropna()
           .map(lambda s: (json.loads(s).get("quote") or {}).get("currency")).dropna())
assert set(monedas.unique()) == {"USD"}, "Verificar la moneda antes de unir cotizaciones."
precios = pd.DataFrame({"Precio en el anuncio": d["price"], "Precio por noche (USD)": precio})
display(precios.head())

# Si algún precio no cuadra, miramos ese anuncio.
cotizado = pd.to_numeric(original.loc[d.index, "price_quote_price_per_night"], errors="coerce")
revisar = ((precio.isna() & (texto_precio.notna() | cotizado.gt(0))) |
           (precio - cotizado).abs().gt(.010001))
if revisar.any():
    display(precios.assign(Cotización=cotizado).loc[revisar].head())

,Precio en el anuncio,Precio por noche (USD)
0,$316.00,316.0
1,$146.50,146.5
2,$121.00,121.0
3,NaN,NaN
4,$282.00,282.0


## 4. Fechas y datos faltantes

Calculamos cuántos días pasaron entre la última reseña y la fecha de captura. Si un anuncio no tiene calificación, guardamos un 0 y lo marcamos para dejarlo fuera del promedio.

In [9]:
corte = pd.to_datetime(d["last_scraped"], format="%Y-%m-%d", errors="raise")
ultima_resena = pd.to_datetime(d["last_review"], format="%Y-%m-%d", errors="raise")
dias = (corte - ultima_resena).dt.days
if corte.isna().any() or dias.dropna().lt(0).any():
    raise ValueError("Hay una fecha de corte ausente o una reseña posterior al corte.")
base["fecha_corte"] = corte.dt.strftime("%Y-%m-%d")
base["tiene_resena_fechada"] = dias.notna().astype("int8")
base["dias_desde_resena"] = dias.fillna(0).astype(int)

calificacion = pd.to_numeric(d["review_scores_rating"], errors="raise")
if not calificacion.dropna().between(1, 5).all():
    raise ValueError("Revisar calificaciones fuera de la escala de 1 a 5.")
base["tiene_calificacion"] = calificacion.notna().astype("int8")
base["calificacion"] = calificacion.fillna(0)
resenas = d["number_of_reviews"].gt(0).map({True: "Con reseñas", False: "Sin reseñas"})
puntajes = calificacion.notna().map({True: "Con calificación", False: "Sin calificación"})
display(pd.crosstab(resenas, puntajes, rownames=["Anuncios"], colnames=["Calificación"]))

Calificación,Con calificación,Sin calificación
Anuncios,,
Con reseñas,11099,0
Sin reseñas,0,2154


Guardamos cerca del 20 % de los anfitriones para probar el modelo más adelante. Con el resto calculamos la mediana por tipo de habitación para completar los datos que faltan. Marcamos cada valor que completamos.

In [10]:
def es_anfitrion_referencia(host_id):
    codigo = hashlib.sha256(host_id.encode("utf-8")).hexdigest()
    return int(codigo[:8], 16) % 5 != 0

referencia = base["host_id"].map(es_anfitrion_referencia)
base["grupo_anfitrion"] = np.where(referencia, "referencia", "reserva")

directas = {"latitude": "latitud", "longitude": "longitud", "accommodates": "capacidad",
            "availability_365": "dias_disponibles_365", "number_of_reviews": "n_resenas"}
for fuente, destino in directas.items():
    base[destino] = pd.to_numeric(d[fuente], errors="raise")

limites = {"latitud": (-90, 90), "longitud": (-180, 180), "capacidad": (1, np.inf),
           "dias_disponibles_365": (0, 365), "n_resenas": (0, np.inf)}
for col, (minimo, maximo) in limites.items():
    if not base[col].between(minimo, maximo).all():
        raise ValueError(f"Revisar rango o faltantes en {col}.")

for fuente, destino in {"bedrooms": "habitaciones", "beds": "camas",
                        "minimum_nights": "noches_minimas"}.items():
    base[destino] = pd.to_numeric(d[fuente], errors="coerce")

imputables = ["habitaciones", "camas", "banos", "noches_minimas"]
balance_nulos = []
for col in imputables:
    minimo = 1 if col == "noches_minimas" else 0
    x = base[col].where(base[col].ge(minimo))
    falta = x.isna()
    # Elegimos un valor que ya aparece en los datos.
    por_tipo = x[referencia].groupby(base.loc[referencia, "tipo_habitacion"]).quantile(.5, interpolation="nearest")
    global_ref = x[referencia].quantile(.5, interpolation="nearest")
    relleno = base["tipo_habitacion"].map(por_tipo).fillna(global_ref)
    base[col + "_imputado"] = falta.astype("int8")
    base[col] = x.fillna(relleno)
    if col != "banos":
        base[col] = base[col].astype(int)
    balance_nulos.append([col, int(falta.sum()), int(base[col].isna().sum())])

display(pd.DataFrame(balance_nulos, columns=["campo", "sustituciones", "nulos finales"]))
display(base.groupby("grupo_anfitrion").agg(anfitriones=("host_id", "nunique"), anuncios=("id", "size")))

,campo,sustituciones,nulos finales
0,habitaciones,24,0
1,camas,373,0
2,banos,51,0
3,noches_minimas,82,0


,anfitriones,anuncios
grupo_anfitrion,,
referencia,4784,10410
reserva,1198,2843
